# Pipeline run — `D:/jeff`

Runs the LBM Suite2p / Cellpose pipeline on every z-plane in `D:/jeff` with the parameter set below.

**Cellpose**: anatomical mode 4 (max_proj), `niter=200`, diameter 2 px, `cellprob_threshold=-6`, `spatial_hp_cp=3`.

**Registration**: enabled, two-step.

**Extraction**: `lam_percentile=0`, `min_neuropil_pixels=0`, `max_overlap=1.0` (very permissive — keeps every detected ROI through extraction).

In [1]:
from pathlib import Path

import lbm_suite2p_python as lsp
from mbo_utilities import imread

input_dir = Path("D:/jeff/single_hemisphere")
output_dir = Path("D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3")
output_dir.mkdir(parents=True, exist_ok=True)

In [2]:
ops = {
    # cellpose
    "anatomical_only": 4,        # 4 = max_proj input to cellpose
    "diameter": 2,               # cell diameter in pixels
    "cellprob_threshold": -6,    # very permissive
    "spatial_hp_cp": 3,          # spatial high-pass before cellpose
    "niter": 200,                # cellpose iterations

    # registration
    "do_registration": 1,
    "two_step_registration": 1,

    # extraction (permissive — keep all detections)
    "lam_percentile": 0,
    "min_neuropil_pixels": 0,
    "max_overlap": 0.99,
}
ops

{'anatomical_only': 4,
 'diameter': 2,
 'cellprob_threshold': -6,
 'spatial_hp_cp': 3,
 'niter': 200,
 'do_registration': 1,
 'two_step_registration': 1,
 'lam_percentile': 0,
 'min_neuropil_pixels': 0,
 'max_overlap': 0.99}

In [3]:
arr = imread(input_dir)
print(f"Input shape: {arr.shape}, dims: {getattr(arr, 'dims', None)}")

Counting frames:   0%|          | 0/9 [00:00<?, ?it/s]

Input shape: (7596, 1, 30, 1002, 725), dims: ('T', 'C', 'Z', 'Y', 'X')


In [4]:
results = lsp.pipeline(
    input_data=arr,
    save_path=str(output_dir),
    ops=ops,
    planes=None,           # all z-planes
    keep_reg=True,
    keep_raw=False,
    force_reg=True,
    force_detect=True,
    # dff_percentile=20,
    reader_kwargs={
        "fix_phase": True,
        "use_fft": True,
    },
)
results

Delegating to run_volume (volumetric input detected)...
Processing 30 planes in volume (Total planes: 30)
Output: D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3

--- Volume Step: Plane 1 ---
Importing suite2p packages...
Writing binary to D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane01_tp00001-07596...


pynwb not installed, save_nwb, read_nwb, and nwb_to_binary will not work. Install with: pip install pynwb


  Computing dF/F...
  Computing ROI statistics...
Plotting results for 395 accepted / 126 rejected ROIs

--- Volume Step: Plane 2 ---
Importing suite2p packages...
Writing binary to D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane02_tp00001-07596...
  Computing dF/F...
  Computing ROI statistics...
Plotting results for 510 accepted / 176 rejected ROIs

--- Volume Step: Plane 3 ---
Importing suite2p packages...
Writing binary to D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane03_tp00001-07596...
  Computing dF/F...
  Computing ROI statistics...
Plotting results for 224 accepted / 102 rejected ROIs

--- Volume Step: Plane 4 ---
Importing suite2p packages...
Writing binary to D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane04_tp00001-07596...
  Computing dF/F...
  Computing ROI statistics...
Plotting results for 656 accepted / 196 rejected ROIs

--- Volume Step: Plane 5 ---

[WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zplane01_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zplane02_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zplane03_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zplane04_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zplane05_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zplane06_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zplane07_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3/zp

In [5]:
 # Reuse the registered output from the previous run; only re-run detection.
prev_output_dir = output_dir  # the dir written above
new_output_dir = Path("D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6")
new_output_dir.mkdir(parents=True, exist_ok=True)

reg_arr = imread(prev_output_dir)  # loads zplane*/data.bin as a
print(f"Reloaded shape: {reg_arr.shape}")

new_ops = {
  **ops,
  "do_registration": 0,        # skip — reuse cached registration
  "anatomical_only": 0,        # functional detection, not
  "tau": 1.3,
  "threshold_scaling": -6,
}

results2 = lsp.pipeline(
  input_data=reg_arr,
  save_path=str(new_output_dir),
  ops=new_ops,
  planes=None,
  keep_reg=True,
  keep_raw=False,
  force_reg=False,             # don't re-register
  force_detect=True,           # re-run detection even though
)
results2

Reloaded shape: (7596, 1, 30, 1002, 725)
Delegating to run_volume (volumetric input detected)...
Processing 30 planes in volume (Total planes: 30)
Output: D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6

--- Volume Step: Plane 1 ---
Importing suite2p packages...
  Copying ops.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane01_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6\zplane01_tp00001-07596
  Copying stat.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane01_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6\zplane01_tp00001-07596
  Copying iscell.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane01_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6\zplane01_tp00001-07596
  Copying F.npy from D:\jeff\single_hemisphere\niter-200_iampercen

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-3.4149063e+11..1.0].


Plotting results for 487 accepted / 2441 rejected ROIs


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-3.4149063e+11..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-3.4149063e+11..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-341490630655.77814..1.0].



--- Volume Step: Plane 2 ---
Importing suite2p packages...
  Copying ops.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane02_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6\zplane02_tp00001-07596
  Copying stat.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane02_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6\zplane02_tp00001-07596
  Copying iscell.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane02_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6\zplane02_tp00001-07596
  Copying F.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane02_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-0_tau-1p3_threshscale-neg6\zplane02_tp00001-07596
  Copying Fneu.npy from D:\jeff\single_hemisphere\niter-200_iampercent

...

Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.489487e+11..1.0].


Plotting results for 810 accepted / 3462 rejected ROIs


Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.489487e+11..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-1.489487e+11..1.0].
Clipping input data to the valid range for imshow with RGB data ([0..1] for floats or [0..255] for integers). Got range [-148948697087.5..1.0].



Genering volumetric statistics...


[WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane01_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane02_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane03_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane04_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane05_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane06_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane07_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane08_tp00001-07596/ops.npy'),
 WindowsPath('D:/jeff/single_hemisphere/anatomical-0_tau-1p3_threshscale-neg6/zplane09_tp00001-07596/ops


## Suite2p defaults vs christians zplane02 settings

| Parameter | Suite2p default | zplane02 value |
|---|---|---|
| `anatomical_only` | 0 | 4 |
| `cellprob_threshold` | 0 | -6 |
| `chan2_thres` | 0.65 | 0.1 |
| `diameter` | 0 | 6.07651 |
| `flow_threshold` | 0.4 | 0 |
| `lam_percentile` | 50 | 0 |
| `max_overlap` | 0.75 | 1 |
| `min_neuropil_pixels` | 350 | 0 |
| `spatial_hp_cp` | 0 | 3 |
| `spatial_scale` | 0 | 1 |
| `suite2p_version` | `1.0.5` | *missing* |
| `tau` | 1 | 1.3 |
| `two_step_registration` | False | 1 |


In [7]:
prev_output_dir = output_dir  # the dir written above
new_output_dir = Path("D:/jeff/single_hemisphere/anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0")
new_output_dir.mkdir(parents=True, exist_ok=True)

reg_arr = imread(prev_output_dir)  # loads zplane*/data.bin as a
print(f"Reloaded shape: {reg_arr.shape}")

new_ops = {
    "diameter": None,
    "do_registration": 0,        # skip — reuse cached registration
    "anatomical_only": 4,        # functional detection, not
    "spatial_hp_cp": 3,
    "spatial_scale": 1,
    "min_neuropil_pixels": 0,
    "lam_percentile": 0,
    "cellprob_threshold": -6,        # functional detection, not
    "flow_threshold": 0,
    "tau": 1.3,
    "max_overlap": 0.99,
}

results2 = lsp.pipeline(
  input_data=reg_arr,
  save_path=str(new_output_dir),
  ops=new_ops,
  planes=None,
  keep_reg=True,
  keep_raw=False,
  force_reg=False,             # don't re-register
  force_detect=True,           # re-run detection even though
)
results2

Reloaded shape: (7596, 1, 30, 1002, 725)
Delegating to run_volume (volumetric input detected)...
Processing 30 planes in volume (Total planes: 30)
Output: D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0

--- Volume Step: Plane 1 ---
Importing suite2p packages...
  Copying ops.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane01_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane01_tp00001-07596
  Copying stat.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane01_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane01_tp00001-07596
  Copying iscell.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane01_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale

Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1513, in run_plane_bin
    if ops["diameter"] is not None and np.isnan(ops["diameter"]):
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1123, in run_volume
    ops_file = run_plane(
               ^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Use

  Copying detect_outputs.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane02_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane02_tp00001-07596
  Copying roi_stats.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane02_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane02_tp00001-07596
Error in run_plane_bin: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
ERROR processing plane 2: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

--- Volume Step: Plane 3 ---
Importing suite2p packages...
  Copying ops.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane03_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scal

Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1513, in run_plane_bin
    if ops["diameter"] is not None and np.isnan(ops["diameter"]):
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1123, in run_volume
    ops_file = run_plane(
               ^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Use

Error in run_plane_bin: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
ERROR processing plane 3: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

--- Volume Step: Plane 4 ---
Importing suite2p packages...
  Copying ops.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane04_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane04_tp00001-07596
  Copying stat.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane04_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane04_tp00001-07596
  Copying iscell.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane04_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-thre

...

Error in run_plane_bin: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
ERROR processing plane 29: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

--- Volume Step: Plane 30 ---
Importing suite2p packages...
  Copying ops.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane30_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane30_tp00001-07596
  Copying stat.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane30_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-threshold0\zplane30_tp00001-07596
  Copying iscell.npy from D:\jeff\single_hemisphere\niter-200_iampercentile-0_min-neuropil_0_spatialhpcp-3\zplane30_tp00001-07596 -> D:\jeff\single_hemisphere\anatomical-4_tau-1p3_cellprob-n6_spatial-scale-1_flow-th

Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1513, in run_plane_bin
    if ops["diameter"] is not None and np.isnan(ops["diameter"]):
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1123, in run_volume
    ops_file = run_plane(
               ^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Use

Error in run_plane_bin: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
ERROR processing plane 30: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1513, in run_plane_bin
    if ops["diameter"] is not None and np.isnan(ops["diameter"]):
                                       ^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()
Traceback (most recent call last):
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 1123, in run_volume
    ops_file = run_plane(
               ^^^^^^^^^^
  File "C:\Users\RBO\repos\mbo_utilities\.venv\Lib\site-packages\lbm_suite2p_python\run_lsp.py", line 2507, in run_plane
    processed = run_plane_bin(ops)
                ^^^^^^^^^^^^^^^^^^
  File "C:\Use

RuntimeError: run_volume failed: All planes resulted in exceptions during processing.

In [ ]:
results = lsp.pipeline(
    input_data=arr,
    save_path=str(output_dir),
    ops=ops,
    planes=None,           # all z-planes
    keep_reg=True,
    keep_raw=False,
    force_reg=True,
    force_detect=True,
    # dff_percentile=20,
    reader_kwargs={
        "fix_phase": True,
        "use_fft": True,
    },
)
results

## Output

Per-plane results land in `D:/jeff/results/zplane0N_tp{first}-{last}/` containing `ops.npy`, `stat.npy`, `F.npy`, `Fneu.npy`, `spks.npy`, `iscell.npy`, and `data.bin`.

Reload by opening `D:/jeff/results` in MBO Studio (`uv run mbo D:/jeff/results`) — every plane comes back as a registered binary you can re-detect / re-classify on without re-registering.